# K=3 soft-membership three-model experiment — v1.1

This notebook trains **Soft Logistic Regression**, an **Embedding MLP**, and **ResNet18 with Layer 4 unfrozen** under the same chronological and soft-label protocol. Class imbalance is handled in every trainer with training-only inverse soft-membership-mass weights. Run the cells in order.


In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('A GPU is strongly recommended for embedding extraction and Layer-4 training.')


In [ ]:
!nvidia-smi


## Discover the attached dataset and package

The dataset is identified by its validated soft-membership manifests. The package is identified by the v1.1 `soft_supervised/run_all_models.py` workflow.


In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
dataset_candidates = []
package_candidates = []
for path in input_root.rglob('*'):
    if path.name == 'train_soft_shuffled.csv' and path.parent.name == 'manifests':
        soft_dir = path.parent.parent
        dataset_candidates.append(soft_dir.parent if soft_dir.name == 'soft_membership' else soft_dir)
    if path.name == 'run_all_models.py' and path.parent.name == 'soft_supervised':
        candidate = path.parent.parent
        if (candidate / 'configs' / 'training_config.json').is_file():
            package_candidates.append(candidate)

dataset_candidates = sorted(set(dataset_candidates))
package_candidates = sorted(set(package_candidates))
print('Dataset candidates:')
for i, path in enumerate(dataset_candidates): print(i, path)
print('\nPackage candidates:')
for i, path in enumerate(package_candidates): print(i, path)
if not dataset_candidates or not package_candidates:
    raise RuntimeError('Attach both the processed dataset and the v1.1 three-model package.')


If more than one candidate appears, change the two indices below before continuing.

In [ ]:
DATASET_ROOT = dataset_candidates[0]
PACKAGE_INPUT = package_candidates[0]
print('Selected dataset:', DATASET_ROOT)
print('Selected package:', PACKAGE_INPUT)


## Copy the package to Kaggle working storage and configure it

In [ ]:
import json
import shutil

PACKAGE_WORKING = Path('/kaggle/working/k3_soft_membership_kaggle_three_model_v1_1')
if PACKAGE_WORKING.exists():
    shutil.rmtree(PACKAGE_WORKING)
shutil.copytree(PACKAGE_INPUT, PACKAGE_WORKING)
CONFIG_PATH = PACKAGE_WORKING / 'configs' / 'training_config.json'
SCRIPTS = PACKAGE_WORKING / 'soft_supervised'
RESULTS_ROOT = Path('/kaggle/working/k3_soft_three_model_results_v1_1')
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = json.load(handle)
config['dataset_dir'] = str(DATASET_ROOT)
config['soft_membership_dir'] = ''
config['output_dir'] = str(RESULTS_ROOT)
config['device'] = 'auto'
config['num_workers'] = 2
with CONFIG_PATH.open('w', encoding='utf-8') as handle:
    json.dump(config, handle, indent=2)
print(json.dumps(config, indent=2))


## Preflight check

This validates paths, manifests, split boundaries, weights, and all three output heads.

In [ ]:
!python -u "{SCRIPTS / 'check_setup.py'}"


## Train all models and select checkpoints using validation only

This cell caches frozen ResNet18 embeddings once, trains Soft Logistic Regression and the MLP on those shared embeddings, then trains ResNet18 Layer 4 from images. It does **not** evaluate test performance.


In [ ]:
!python -u "{SCRIPTS / 'run_all_models.py'}"


## Inspect the validation-only comparison

In [ ]:
import pandas as pd

validation_comparison = pd.read_csv(RESULTS_ROOT / 'validation_model_comparison.csv')
display(validation_comparison)


## Independent test evaluation — run once

Run this only after the model definitions, hyperparameters, and validation selection are final. The script evaluates all three validation-best checkpoints and writes a lock that prevents an accidental repeat.


In [ ]:
!python -u "{SCRIPTS / 'evaluate_all_models.py'}"


In [ ]:
test_comparison = pd.read_csv(RESULTS_ROOT / 'independent_test_model_comparison.csv')
display(test_comparison)


## Archive the complete results

In [ ]:
import shutil
from IPython.display import FileLink

archive_base = Path('/kaggle/working/k3_soft_three_model_results')
archive_path = shutil.make_archive(str(archive_base), 'zip', RESULTS_ROOT)
print('Created:', archive_path)
FileLink(archive_path)
